# TABA H9ETS — Report Charts (FEMTO/PRONOSTIA Green AI)

Self-contained notebook that regenerates the 3 charts used in the report:
1. Pareto scatter — energy vs. critical recall / lead time
2. Model size by algorithm
3. R² across the methodology fixes (diagnostic journey)

The underlying numbers are embedded directly below (from `final_results_aggregated.csv` and the diagnostic runs), so this notebook does **not** need the raw FEMTO dataset to run — just run all cells top to bottom and download the PNGs at the end.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import io

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.edgecolor": "#c3c2b7",
    "axes.labelcolor": "#0b0b0b",
    "text.color": "#0b0b0b",
    "xtick.color": "#52514e",
    "ytick.color": "#52514e",
    "axes.grid": True,
    "grid.color": "#e1e0d9",
    "grid.linewidth": 0.7,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})

SERIES = {"half-rate": "#2a78d6", "reference": "#eb6834", "top-10-features": "#1baf7a"}
MARKERS = {"Ridge": "o", "HistGradientBoosting": "s", "RandomForest": "^", "MLP": "D"}

## Data — aggregated results (mean across the 3 operating conditions)

This is `taba_femto_outputs/final_results_aggregated.csv` from the project repo, embedded inline. If you re-run the full pipeline and get new numbers, just replace this CSV block.

In [ ]:
AGGREGATED_CSV = """strategy,model,mae_frac,r2,critical_recall,lead_time_min,train_energy_kwh,co2_g,model_size_kb,n_features
half-rate,HistGradientBoosting,0.23004760185571962,-0.05006812506820146,0.3019817073170732,20.38888888888889,1.2272920243267673e-06,0.00035739318146689865,358.1975911458333,24
half-rate,MLP,0.2662906369805475,-0.41288283960514055,0.3178397212543554,28.88888888888889,9.23787228545908e-07,0.0002689039577962516,95.87434895833333,24
half-rate,RandomForest,0.22319622434656017,-0.016218679052250867,0.31626016260162604,20.38888888888889,7.792230457800808e-06,0.0023576569491374605,37656.7548828125,24
half-rate,Ridge,0.323103407820857,-1.08460029222025,0.26343931475029037,18.72222222222222,9.460829432602449e-10,4.719532405078957e-07,0.6181640625,24
reference,HistGradientBoosting,0.2828679136131797,-0.5927678593906032,0.4001974448315912,23.0,1.6008812324766209e-06,0.0004665114183023304,358.3929036458333,24
reference,MLP,0.5401337065683451,-4.197246541294894,0.29412746806039486,30.055555555555557,8.89411366799735e-07,0.0002588933981095928,96.03125,24
reference,RandomForest,0.27979954798207723,-0.6020818230032244,0.4060917537746806,23.0,7.878132318946091e-06,0.002379760989335277,37656.7548828125,24
reference,Ridge,0.6327898053691093,-5.506579263551861,0.40985772357723577,20.666666666666668,6.00842304864753e-08,1.7881162118429667e-05,0.6181640625,24
top-10-features,HistGradientBoosting,0.3048787786438225,-0.785497366968743,0.4318031358885017,23.0,1.1487453139622276e-06,0.0003350213896392662,329.0999348958333,10
top-10-features,MLP,0.3221628149755739,-0.878641536921625,0.3029108594657375,17.61111111111111,1.079824282495786e-06,0.00031429325672850585,75.26595052083333,10
top-10-features,RandomForest,0.30455467772780426,-0.8138666580298174,0.43281939605110337,23.0,3.844296742499205e-06,0.001223118509607235,37656.7548828125,10
top-10-features,Ridge,0.5426320115265953,-3.4312973775262634,0.39410569105691057,20.61111111111111,1.803487370872601e-09,8.127512826201436e-07,0.5087890625,10
"""

df = pd.read_csv(io.StringIO(AGGREGATED_CSV))
df

## Chart 1 — Pareto scatter: Energy × Lead time / Critical recall

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.3))

for ax, ykey, ylabel, yfmt in [
    (axes[0], "lead_time_min", "Lead time (min)", None),
    (axes[1], "critical_recall", "Critical recall", "pct"),
]:
    for _, row in df.iterrows():
        ax.scatter(row["train_energy_kwh"], row[ykey],
                   color=SERIES[row["strategy"]], marker=MARKERS[row["model"]],
                   s=90, edgecolor="white", linewidth=1.2, zorder=3)
    ax.set_xscale("log")
    ax.set_xlabel("Training energy (kWh, log scale)")
    ax.set_ylabel(ylabel)
    if yfmt == "pct":
        ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.set_ylim(bottom=0)
    ax.grid(True, which="major", axis="both")

strategy_handles = [plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=c, markersize=9, label=s)
                     for s, c in SERIES.items()]
model_handles = [plt.Line2D([0], [0], marker=m, color="#52514e", linestyle="", markersize=8, label=k)
                  for k, m in MARKERS.items()]
fig.legend(handles=strategy_handles, title="Strategy (color)", loc="upper center",
           bbox_to_anchor=(0.28, 0.02), ncol=1, frameon=False, fontsize=9, title_fontsize=9)
fig.legend(handles=model_handles, title="Model (shape)", loc="upper center",
           bbox_to_anchor=(0.72, 0.02), ncol=1, frameon=False, fontsize=9, title_fontsize=9)
fig.suptitle("Energy vs. predictive-maintenance viability (averaged across 3 conditions)", fontsize=12, y=1.02)
plt.tight_layout(rect=[0, 0.14, 1, 1])
plt.savefig("chart1_pareto.png", dpi=200, bbox_inches="tight")
plt.show()

## Chart 2 — Model size by algorithm (reference configuration)

In [ ]:
size_df = df[df["strategy"] == "reference"].sort_values("model_size_kb")
fig, ax = plt.subplots(figsize=(7, 4))
colors = ["#2a78d6", "#eb6834", "#1baf7a", "#e34948"]
bars = ax.bar(size_df["model"], size_df["model_size_kb"], color=colors[:len(size_df)])
ax.set_yscale("log")
ax.set_ylabel("Serialized model size (KB, log scale)")
ax.set_title("Model size by algorithm (reference configuration)")
for b, v in zip(bars, size_df["model_size_kb"]):
    label = f"{v:,.0f} KB" if v >= 1 else f"{v:.2f} KB"
    ax.annotate(label, (b.get_x() + b.get_width() / 2, v), xytext=(0, 5),
                textcoords="offset points", ha="center", fontsize=9)
plt.xticks(rotation=10)
plt.tight_layout()
plt.savefig("chart2_model_size.png", dpi=200, bbox_inches="tight")
plt.show()

## Chart 3 — R² across the methodology fixes (diagnostic journey)

These four values are held-out results from the diagnostics documented in the report (Section 3.4), using **RandomForest** consistently across all four to isolate the effect of each fix (rather than mixing model changes into it):

1. **Same-bearing holdout** (Bearing1_1, random row split) — sanity check that the extracted features carry signal at all.
2. **Cross-condition split, absolute RUL** — the original (broken) experiment: trained on conditions 2+3, tested on condition 1.
3. **Within-condition split, absolute RUL** — condition 2, trained on one bearing, tested on its sibling, still using raw minutes as the target.
4. **Within-condition split, normalized RUL** — same split as #3, but target changed to `rul_frac` (fraction of life remaining).

If you re-run the diagnostics yourself and get different numbers, just edit the `values` list below.

In [ ]:
stages = [
    "Same-bearing\nholdout\n(sanity check)",
    "Cross-condition split\nabsolute RUL\n(original attempt)",
    "Within-condition split\nabsolute RUL\n(condition 2)",
    "Within-condition split\nnormalized RUL\n(condition 2)",
]
values = [0.993, -0.303, 0.307, 0.615]
colors = ["#1baf7a" if v >= 0 else "#e34948" for v in values]

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(stages, values, color=colors, width=0.6)
ax.axhline(0, color="#898781", linewidth=1)
ax.set_ylabel("R² (RandomForest)")
ax.set_title("R² across the methodology fixes (RandomForest held constant)")
for b, v in zip(bars, values):
    ax.annotate(f"{v:.2f}", (b.get_x() + b.get_width() / 2, v),
                xytext=(0, 6 if v >= 0 else -14), textcoords="offset points",
                ha="center", fontsize=10, fontweight="bold")
plt.xticks(fontsize=9)
plt.tight_layout()
plt.savefig("chart3_diagnostic_journey.png", dpi=200, bbox_inches="tight")
plt.show()

## Download the PNGs

Run this cell in Colab to download all three charts to your computer.

In [ ]:
from google.colab import files

for fname in ["chart1_pareto.png", "chart2_model_size.png", "chart3_diagnostic_journey.png"]:
    files.download(fname)